# Negation Augmentation Test

**Goal:** Test negation patterns on sample clauses from the LexGLUE unfair_tos dataset to verify the augmentation logic before full implementation.


## 1. Setup & Installation

In [ ]:
# Install required libraries
!pip install -q datasets

In [ ]:
# Import libraries
from datasets import load_dataset
import re

print("✓ Libraries imported successfully!")

## 2. Define Negation Function (Refined)

In [ ]:
def has_existing_negation(clause):
    """
    Check if clause already contains negation words to avoid double negatives.
    Returns True if negation is found.
    """
    clause_lower = clause.lower()
    negation_words = [
        r'\bnot\b', r'\bno\b', r'\bnever\b', r'\bneither\b', 
        r'\bnor\b', r'\bnone\b', r'\bnothing\b', r'\bnobody\b',
        r'\bcannot\b', r"\bcan't\b", r"\bwon't\b", r"\bshouldn't\b",
        r"\bwouldn't\b", r"\bdidn't\b", r"\bdoesn't\b", r"\bdon't\b",
        r"\bisn't\b", r"\baren't\b", r"\bwasn't\b", r"\bweren't\b"
    ]
    
    for pattern in negation_words:
        if re.search(pattern, clause_lower):
            return True
    return False

def negate_clause(clause):
    """
    Apply negation to a risky clause.
    Returns (negated_clause, pattern_used) or (None, None) if negation doesn't make sense.
    """
    # Skip clauses that already have negations to avoid double negatives
    if has_existing_negation(clause):
        return None, None
    
    clause_lower = clause.lower()
    
    # Pattern 1: Modal verb "may"
    if ' may ' in clause_lower:
        negated = re.sub(r'\bmay\b', 'may not', clause, count=1, flags=re.IGNORECASE)
        return negated, "Modal: may → may not"
    
    # Pattern 2: Modal verb "can"
    elif ' can ' in clause_lower:
        negated = re.sub(r'\bcan\b', 'cannot', clause, count=1, flags=re.IGNORECASE)
        return negated, "Modal: can → cannot"
    
    # Pattern 3: Modal verb "will"
    elif ' will ' in clause_lower:
        negated = re.sub(r'\bwill\b', 'will not', clause, count=1, flags=re.IGNORECASE)
        return negated, "Modal: will → will not"
    
    # Pattern 4: Modal verb "shall"
    elif ' shall ' in clause_lower:
        negated = re.sub(r'\bshall\b', 'shall not', clause, count=1, flags=re.IGNORECASE)
        return negated, "Modal: shall → shall not"
    
    # Pattern 5: "reserve the right"
    elif 'reserve the right' in clause_lower:
        negated = re.sub(r'reserve the right', 'do not reserve the right', clause, count=1, flags=re.IGNORECASE)
        return negated, "Phrase: reserve the right → do not reserve the right"
    
    # Pattern 6: "has the right"
    elif 'has the right' in clause_lower:
        negated = re.sub(r'has the right', 'does not have the right', clause, count=1, flags=re.IGNORECASE)
        return negated, "Phrase: has the right → does not have the right"
    
    # Pattern 7: "have the right"
    elif 'have the right' in clause_lower:
        negated = re.sub(r'have the right', 'do not have the right', clause, count=1, flags=re.IGNORECASE)
        return negated, "Phrase: have the right → do not have the right"
    
    # Pattern 8: "is/are entitled to"
    elif 'is entitled to' in clause_lower or 'are entitled to' in clause_lower:
        negated = re.sub(r'is entitled to', 'is not entitled to', clause, count=1, flags=re.IGNORECASE)
        negated = re.sub(r'are entitled to', 'are not entitled to', negated, count=1, flags=re.IGNORECASE)
        return negated, "Phrase: entitled to → not entitled to"
    
    # Pattern 9: "agree to" / "agrees to"
    elif 'agree to' in clause_lower or 'agrees to' in clause_lower:
        negated = re.sub(r'agree to', 'do not agree to', clause, count=1, flags=re.IGNORECASE)
        negated = re.sub(r'agrees to', 'does not agree to', negated, count=1, flags=re.IGNORECASE)
        return negated, "Phrase: agree to → do not agree to"
    
    return None, None

print("✓ Negation function defined with 9 patterns")
print("✓ Added filter to skip clauses with existing negations")

## 3. Load Dataset

In [ ]:
print("Loading LexGLUE unfair_tos dataset...")
dataset = load_dataset("lex_glue", "unfair_tos")
train_data = dataset['train']

# Get label names
label_names = train_data.features['labels'].feature.names

print(f"\n✓ Dataset loaded successfully!")
print(f"  - Training examples: {len(train_data)}")
print(f"  - Risk categories: {len(label_names)}")
print(f"\nRisk Categories:")
for i, label in enumerate(label_names):
    print(f"  {i+1}. {label}")

## 4. Test Negation on Sample Clauses

In [ ]:
# Number of samples to test
NUM_SAMPLES = 20

print("=" * 100)
print(f"TESTING NEGATION ON {NUM_SAMPLES} RISKY CLAUSES (WITH NEGATION FILTER)")
print("=" * 100)

successful_negations = 0
failed_negations = 0
skipped_existing_negation = 0

# Test on risky clauses
for i, example in enumerate(train_data):
    if successful_negations >= NUM_SAMPLES:
        break
    
    # Only process clauses with at least one risk label
    if len(example['labels']) > 0:
        original_clause = example['text']
        original_labels = [label_names[idx] for idx in example['labels']]
        
        # Check if clause has existing negation
        if has_existing_negation(original_clause):
            skipped_existing_negation += 1
            continue
        
        # Try to negate
        negated_clause, pattern = negate_clause(original_clause)
        
        if negated_clause:
            successful_negations += 1
            print(f"\n{'─' * 100}")
            print(f"EXAMPLE {successful_negations}")
            print(f"{'─' * 100}")
            print(f"Pattern Used: {pattern}")
            print(f"\nOriginal Risks: {', '.join(original_labels)}")
            print(f"\nORIGINAL CLAUSE:")
            print(f"  {original_clause}")
            print(f"\nNEGATED CLAUSE (Safe):")
            print(f"  {negated_clause}")
        else:
            failed_negations += 1

print(f"\n\n{'=' * 100}")
print(f"Skipped {skipped_existing_negation} clauses with existing negations (to avoid double negatives)")
print(f"{'=' * 100}")

## 5. Summary Statistics

In [ ]:
print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)
print(f"✓ Successful negations: {successful_negations}")
print(f"✗ Failed negations (no pattern matched): {failed_negations}")
print(f"⊘ Skipped (existing negation): {skipped_existing_negation}")
total_processed = successful_negations + failed_negations
if total_processed > 0:
    print(f"\nSuccess rate (excluding skipped): {successful_negations / total_processed * 100:.1f}%")

## 6. Pattern Coverage Analysis

In [ ]:
print("\n" + "=" * 100)
print("PATTERN COVERAGE ANALYSIS")
print("=" * 100)

# Analyze which patterns are most common
pattern_counts = {}
total_risky = 0
total_with_existing_negation = 0

for example in train_data:
    if len(example['labels']) > 0:
        total_risky += 1
        
        # Check for existing negation
        if has_existing_negation(example['text']):
            total_with_existing_negation += 1
            continue
        
        _, pattern = negate_clause(example['text'])
        if pattern:
            pattern_counts[pattern] = pattern_counts.get(pattern, 0) + 1

print(f"\nTotal risky clauses in dataset: {total_risky}")
print(f"Clauses with existing negations (skipped): {total_with_existing_negation} ({total_with_existing_negation/total_risky*100:.1f}%)")
print(f"Clauses that can be negated: {sum(pattern_counts.values())} ({sum(pattern_counts.values())/total_risky*100:.1f}%)")
print(f"\nPattern frequency:")
for pattern, count in sorted(pattern_counts.items(), key=lambda x: x[1], reverse=True):
    percentage = (count / total_risky) * 100
    print(f"  {pattern}: {count} clauses ({percentage:.1f}%)")

## 7. Quality Check: Verify No Double Negatives

Let's verify that our filter successfully prevents double negatives:

In [ ]:
print("=" * 100)
print("DOUBLE NEGATIVE CHECK")
print("=" * 100)

# Check for double negatives in our negations
double_negative_patterns = [
    r'\bcannot not\b', r'\bcan not not\b', r'\bmay not not\b',
    r'\bwill not not\b', r'\bshall not not\b',
    r'\bnot have no\b', r'\bnot be no\b'
]

double_negatives_found = 0
sample_count = 0

for example in train_data:
    if len(example['labels']) > 0 and sample_count < 1000:
        sample_count += 1
        negated, _ = negate_clause(example['text'])
        if negated:
            for pattern in double_negative_patterns:
                if re.search(pattern, negated, re.IGNORECASE):
                    double_negatives_found += 1
                    print(f"\n⚠️ Double negative found: {negated[:200]}...")
                    break

if double_negatives_found == 0:
    print(f"\n✅ SUCCESS: No double negatives found in {sample_count} sample negations!")
    print("✅ The negation filter is working correctly!")
else:
    print(f"\n❌ Found {double_negatives_found} double negatives in {sample_count} samples")
    print("❌ Filter needs refinement")